In [1]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql
from urllib.parse import quote_plus

In [5]:
user = 'root'
password = '12345'
server = 'localhost'
db_name = 'incident_analytics'
engine = create_engine(f"mysql+pymysql://{user}:{password}@{server}:3306/{db_name}")

In [6]:
b=pd.read_sql(r"select * from clean_incidents",engine)

In [15]:
b

,case_id,created_at,resolved_at,event_steps,variant,priority,issue_type,resolver,report_channel,short_description,csat_score,mttr_hours,sla_breached,created_hour,created_day
0,INC0001,2023-11-17 11:17:00,2023-11-17 21:28:00,7,Variant 4,Medium,Performance Issue,Sarah,Website,Application crash,3,10,0,11,Friday
1,INC0002,2023-08-15 07:08:00,2023-08-15 12:51:00,8,Variant 3,High,Performance Issue,Sarah,App,Data loss issue,3,5,0,7,Tuesday
2,INC0003,2023-11-23 19:27:00,2023-11-24 03:25:00,10,Variant 9,High,Feature Request,Sam,Website,New feature request,4,7,0,19,Thursday
3,INC0004,2023-09-20 18:56:00,2023-09-21 04:30:00,6,Variant 1,Medium,Incident,Sam,Email,Unable to login,4,9,0,18,Wednesday
4,INC0005,2023-07-11 12:47:00,2023-07-12 18:39:00,8,Variant 3,Low,Performance Issue,Sarah,Website,Application crash,3,29,0,12,Tuesday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31583,INC31584,2023-03-21 01:31:00,2023-03-21 04:28:00,4,Variant 2,High,Feature Request,Sam,Email,New feature request,2,2,0,1,Tuesday
31584,INC31585,2023-05-10 08:28:00,2023-05-10 18:51:00,6,Variant 1,Medium,Bug,Sam,Website,Data loss issue,4,10,0,8,Wednesday
31585,INC31586,2023-08-31 03:52:00,2023-08-31 21:40:00,12,Variant 5,Medium,Bug,Sarah,Email,Data loss issue,5,17,0,3,Thursday
31586,INC31587,2023-02-25 17:39:00,2023-02-26 02:55:00,6,Variant 1,Medium,Service Request,Sam,Website,Password reset needed,3,9,0,17,Saturday


In [8]:
b.describe()

,created_at,resolved_at,event_steps,csat_score,mttr_hours,sla_breached,created_hour
count,31588,31588,31588.000000,31588.000000,31588.000000,31588.000000,31588.000000
mean,2023-07-01 13:23:37.311004160,2023-07-02 04:25:31.701912064,7.689661,3.257028,14.540553,0.023237,11.478061
min,2023-01-01 00:07:00,2023-01-01 07:11:00,4.000000,1.000000,0.000000,0.000000,0.000000
25%,2023-04-01 08:23:15,2023-04-01 23:08:45,6.000000,3.000000,8.000000,0.000000,5.000000
50%,2023-06-30 21:21:30,2023-07-01 11:29:30,8.000000,3.000000,12.000000,0.000000,12.000000
75%,2023-09-30 13:33:45,2023-10-01 05:44:45,8.000000,4.000000,19.000000,0.000000,17.000000
max,2023-12-31 23:38:00,2024-01-02 07:44:00,15.000000,5.000000,56.000000,1.000000,23.000000
std,NaN,NaN,2.199076,1.150170,9.014706,0.150657,6.929724


In [14]:
b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31588 entries, 0 to 31587
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   case_id            31588 non-null  object        
 1   created_at         31588 non-null  datetime64[ns]
 2   resolved_at        31588 non-null  datetime64[ns]
 3   event_steps        31588 non-null  int64         
 4   variant            31588 non-null  object        
 5   priority           31588 non-null  object        
 6   issue_type         31588 non-null  object        
 7   resolver           31588 non-null  object        
 8   report_channel     31588 non-null  object        
 9   short_description  31588 non-null  object        
 10  csat_score         31588 non-null  int64         
 11  mttr_hours         31588 non-null  int64         
 12  sla_breached       31588 non-null  int64         
 13  created_hour       31588 non-null  int64         
 14  create

In [11]:
b["created_at"] = pd.to_datetime(["created_at"])

In [13]:
b["resolved_at"]=pd.to_datetime(b["resolved_at"])

In [17]:
print("Total tickets loaded:", len(b))

Total tickets loaded: 31588


In [18]:
b.isnull().sum()

case_id              0
created_at           0
resolved_at          0
event_steps          0
variant              0
priority             0
issue_type           0
resolver             0
report_channel       0
short_description    0
csat_score           0
mttr_hours           0
sla_breached         0
created_hour         0
created_day          0
dtype: int64

In [21]:
b["sla_breached"].value_counts()

sla_breached
0    30854
1      734
Name: count, dtype: int64

In [22]:
b["created_at"].min()

Timestamp('2023-01-01 00:07:00')

In [23]:
b["created_at"].max()

Timestamp('2023-12-31 23:38:00')

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# --- STAGE 2: NLP ROOT-CAUSE CLUSTERING ---
tfidf = TfidfVectorizer(stop_words="english", max_features=200)
text_matrix = tfidf.fit_transform(b["short_description"].fillna("unknown"))

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
b["root_cause_topic"] = "Topic_" + kmeans.fit_predict(text_matrix).astype(str)

print("=== TOP KEYWORDS PER ROOT-CAUSE CLUSTER ===")
terms = tfidf.get_feature_names_out()
for i in range(5):
    top_words = [terms[ind] for ind in kmeans.cluster_centers_.argsort()[:, -5:][i]]
    print(f"Topic_{i}: {top_words}")

# --- STAGE 3: BALANCED SLA BREACH PREDICTION ---
features = ["priority", "issue_type", "resolver", "report_channel", "created_hour", "created_day"]
X = pd.get_dummies(b[features], drop_first=True)
y = b["sla_breached"]

# Stratified split keeps the 2.3% breach ratio equal in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# class_weight="balanced" handles the 97.7% vs 2.3% class imbalance
clf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)

print("\n=== SLA BREACH MODEL REPORT ===")
print(classification_report(y_test, clf.predict(X_test)))

# Add breach probability score (0.0 to 1.0) to every ticket
b["breach_probability"] = clf.predict_proba(X)[:, 1]


=== TOP KEYWORDS PER ROOT-CAUSE CLUSTER ===
Topic_0: ['crash', 'needed', 'request', 'feature', 'new']
Topic_1: ['feature', 'data', 'needed', 'crash', 'application']
Topic_2: ['needed', 'working', 'loss', 'issue', 'data']
Topic_3: ['unable', 'printer', 'working', 'outage', 'server']
Topic_4: ['crash', 'needed', 'working', 'glitch', 'ui']

=== SLA BREACH MODEL REPORT ===
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      6171
           1       0.30      0.35      0.32       147

    accuracy                           0.97      6318
   macro avg       0.64      0.67      0.65      6318
weighted avg       0.97      0.97      0.97      6318



In [26]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# --- STAGE 4: 30-DAY TICKET VOLUME FORECAST ---
# Count total tickets created per day
daily = b.set_index("created_at").resample("D").size()

# Train time-series model with 7-day weekly seasonality
ts_model = ExponentialSmoothing(daily, trend="add", seasonal="add", seasonal_periods=7).fit()

# Forecast the next 30 days
forecast = ts_model.forecast(30).reset_index()
forecast.columns = ["forecast_date", "predicted_tickets"]
forecast["predicted_tickets"] = forecast["predicted_tickets"].round(0).astype(int)

# --- STAGE 5: SAVE TO MYSQL & CSV FOR POWER BI ---
b.to_sql("scored_incidents", engine, if_exists="replace", index=False)
forecast.to_sql("ticket_forecast", engine, if_exists="replace", index=False)

b.to_csv("powerbi_incidents_scored.csv", index=False)
forecast.to_csv("powerbi_ticket_forecast.csv", index=False)

print("All done! Exported 'scored_incidents' and 'ticket_forecast' for Power BI.")


All done! Exported 'scored_incidents' and 'ticket_forecast' for Power BI.


In [27]:
topic_names = {
    "Topic_0": "Your Label 0",
    "Topic_1": "Your Label 1",
    "Topic_2": "Your Label 2",
    "Topic_3": "Your Label 3",
    "Topic_4": "Your Label 4"
}
b["root_cause_topic"] = b["root_cause_topic"].map(topic_names)
